> **Research record.** Cleaned copy of `XXX-calculate-WFH/203-validate-home-vs-census-article.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Validates detected home and work locations and their representativeness against official statistics (figures used in the article).
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# Description
The goal of the notebook is to validate detected home and work locations and asses representativeness of their distributions with official data

In [ ]:
import pandas as pd
from geofunctions import utils
import geopandas as gpd
import os


In [ ]:
df = gpd.read_file("../data/to_map/geohash_work_home_stat_monthly.geojson")

### Aggregate number of home locations by stat area 

In [ ]:
gdf_census_gush = gpd.read_file(
    utils.get_path("processed", "adm", "geohash_stat_area_pairs.geojson")
)

In [ ]:
full_df_census = df.merge(
    gdf_census_gush[["geohash", "YISHUV_STAT_2022"]], on="geohash"
)

In [ ]:
statarea_hl = full_df_census.groupby(["YISHUV_STAT_2022", "month"])[
    ["num_homes"]
].sum()

#### Add census data

In [ ]:
census_gushdan = gpd.read_file(
    utils.get_path("processed", "census", "census_gushdan_main_features.geojson")
)

#### Do cross join

In [ ]:
# Create a full list of census tracts from census_gushdan
full_census_tracts = census_gushdan[['YISHUV_STAT_2022','pop_approx','SEMEL_YISHUV','SHEM_YISHUV_HEB']].drop_duplicates()

# Create a full list of unique months from statarea_hl
unique_months = statarea_hl.reset_index()[['month']].drop_duplicates()

# Perform a cross join between census tracts and months
cross_joined = full_census_tracts.merge(unique_months, how='cross')

In [ ]:
statarea_full = cross_joined.merge(statarea_hl.reset_index(), on=["YISHUV_STAT_2022",'month'],how='left')

### Test missing values

In [ ]:
import matplotlib.pyplot as plt

# Filter out rows where pop_approx is null
filtered_data = statarea_full[~statarea_full.pop_approx.isnull()]

# Compute the proportion of missing num_homes values per month
missing_values = filtered_data.groupby('month')["num_homes"].apply(lambda x: x.isnull().mean())

# Create a visually improved plot
plt.figure(figsize=(10, 5))
plt.plot(missing_values.index, missing_values.values, marker='o', linestyle='-', linewidth=2, markersize=6)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Proportion of Missing num_homes", fontsize=12)
plt.title("Proportion of Missing num_homes Over Time", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.show()


In [ ]:
statarea_full["pop_approx"] = statarea_full["pop_approx"].fillna(0)

## Explore dependencies and potential discrepancies

In [ ]:
from sklearn.linear_model import LinearRegression


def reg(df, x, y):
    x = df[x].values.reshape(-1, 1)
    y = df[y].values
    reg = LinearRegression(fit_intercept=True).fit(x, y)
    reg_line = reg.predict(x)
    return reg_line

In [ ]:
statarea_full["predicted_pop"] = (
    statarea_full.groupby("month")
    .apply(
        lambda group: pd.Series(
            reg(group, "num_homes", "pop_approx"), index=group.index
        )
    )
    .reset_index(level=0, drop=True)
)

In [ ]:
statarea_full["residual"] = statarea_full["pop_approx"] - statarea_full["predicted_pop"]

In [ ]:
from matplotlib import pyplot as plt


plt.hist(statarea_full["residual"], bins=100);

### Correlation

In [ ]:
import matplotlib.pyplot as plt

# Extracting the first value of "home_locations" for each month and plotting
statarea_full.groupby("month")[["pop_approx", "num_homes"]].apply(
    lambda group: group[["pop_approx", "num_homes"]].corr().iloc[0, 1]
).plot(figsize=(12, 6), marker="o", linewidth=2, color="teal", legend=False)

# Adding title and labels
plt.title(
    "Census Population vs Home locations by stat area for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(range(len(statarea_full.month.unique())),statarea_full.month.unique().tolist(),rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## By degree of religiosity

In [ ]:
dict_religions = {
    "חילוני": "secular",
    "מסורתי": "traditional",
    "דתי/ דתי מאוד": "religious",
    "חרדי": "haredi",
}
statarea_full["religion_level"] = statarea_full["hh_MidatDatiyut_Name"].map(
    dict_religions
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Extracting the first value of "home_locations" for each month and plotting
correlations = (
    statarea_full.groupby(["month", "religion_level"])[
        ["pop_approx", "num_homes"]
    ]
    .apply(lambda group: group[["pop_approx", "num_homes"]].corr().iloc[0, 1])
    .reset_index(name="correlation")
)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=correlations,
    x="month",
    y="correlation",
    hue="religion_level",
    marker="o",
    linewidth=2,
    color="teal",
    legend=True,
)

# Adding title and labels
plt.title(
    "Census Population vs Home locations by level of religiosity for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=90, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Exclude haredi areas 

In [ ]:
import matplotlib.pyplot as plt

# Extracting the first value of "home_locations" for each month and plotting
correlations = (
     statarea_full[
         statarea_full.religion_level.isin(["secular", "traditional"])
    ]
    .groupby(["month"])[["pop_approx", "num_homes"]]
    .apply(lambda group: group[["pop_approx", "num_homes"]].corr().iloc[0, 1])
    .reset_index(name="correlation")
)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=correlations,
    x="month",
    y="correlation",
    marker="o",
    linewidth=2,
    color="teal",
    legend=True,
)

# Adding title and labels
plt.title(
    "Census Population vs Home locations in cities for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Further analysis is optional

In [ ]:
df_homeloc_pop_stat["flag_not_religious"] = df_homeloc_pop_stat.religion_level.isin(
    ["secular", "traditional","religious"]
)

In [ ]:
annual_home_pop = (
    df_homeloc_pop_stat[(~df_homeloc_pop_stat.month.isin(["201911", "201909"]))]
    .groupby(["year", "YISHUV_STAT_2022", "flag_city", "flag_not_religious"])[
        ["pop_approx", "home_locations"]
    ]
    .mean()
    .reset_index()
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
import math

# Unique years
years = annual_home_pop.year.unique()

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

# Plot for each year
for ax, year in zip(axes, years[:3]):

    regline_city = reg(
        annual_home_pop[
            (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
        ],
        x="home_locations",
        y="pop_approx",
    )
    regline_not_city = reg(
        annual_home_pop[
            (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
        ],
        x="home_locations",
        y="pop_approx",
    )
    y_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
    ]["pop_approx"].values
    y_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
    ]["pop_approx"].values
    x_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
    ]["home_locations"].values
    x_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
    ]["home_locations"].values
    r2_city = r2_score(y_city, regline_city)
    r2_not_city = r2_score(y_not_city, regline_not_city)
    sns.scatterplot(
        data=annual_home_pop[annual_home_pop.year == year],
        y="pop_approx",
        x="home_locations",
        hue="flag_not_religious",
        ax=ax,
        alpha=0.5,
    )
    ax.set_title(str(year), fontsize=12)
    ax.set_ylabel("Population Approximation")
    ax.set_xlabel(
        "Home Locations" if ax == axes[0] else ""
    )  # Add ylabel only to the first subplot
    # Plot the regression line
    ax.plot(
        annual_home_pop[
            (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
        ]["home_locations"],
        regline_city,
        color="orange",
        alpha=0.5,
    )
    ax.text(
        math.floor(max(x_city) * 0.8),
        math.floor(max(regline_city)),
        f"R={round(r2_city,2)}",
        color="orange",
    )
    ax.plot(
        annual_home_pop[
            (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
        ]["home_locations"],
        regline_not_city,
        color="blue",
        alpha=0.5,
    )
    ax.text(
        math.floor(max(x_not_city) * 0.8),
        math.floor(max(regline_not_city)),
        f"R={round(r2_not_city,2)}",
        color="Blue",
    )
# Adjust layout and show
plt.tight_layout()
plt.show()
# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

# Plot for each year
for ax, year in zip(axes, years[3:]):
    regline_city = reg(
        annual_home_pop[
            (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
        ],
        x="home_locations",
        y="pop_approx",
    )
    regline_not_city = reg(
        annual_home_pop[
            (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
        ],
        x="home_locations",
        y="pop_approx",
    )
    y_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
    ]["pop_approx"].values
    y_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
    ]["pop_approx"].values
    x_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
    ]["home_locations"].values
    x_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
    ]["home_locations"].values
    r2_city = r2_score(y_city, regline_city)
    r2_not_city = r2_score(y_not_city, regline_not_city)
    sns.scatterplot(
        data=annual_home_pop[annual_home_pop.year == year],
        y="pop_approx",
        x="home_locations",
        hue="flag_not_religious",
        ax=ax,
        alpha=0.5,
    )
    ax.set_title(str(year), fontsize=12)
    ax.set_ylabel("Population Approximation")
    ax.set_xlabel(
        "Home Locations" if ax == axes[0] else ""
    )  # Add ylabel only to the first subplot
    ax.plot(
        annual_home_pop[
            (annual_home_pop.year == year) & (annual_home_pop.flag_not_religious)
        ]["home_locations"],
        regline_city,
        color="orange",
        alpha=0.5,
    )
    ax.plot(
        annual_home_pop[
            (annual_home_pop.year == year) & (~annual_home_pop.flag_not_religious)
        ]["home_locations"],
        regline_not_city,
        color="blue",
        alpha=0.5,
    )
    ax.text(
        math.floor(max(x_city) * 0.8),
        math.floor(max(regline_city)),
        f"R={round(r2_city,2)}",
        color="orange",
    )
    ax.text(
        math.floor(max(x_not_city) * 0.8),
        math.floor(max(regline_not_city)),
        f"R={round(r2_not_city,2)}",
        color="Blue",
    )
# Adjust layout and show
plt.tight_layout()
plt.show()

### Analyse home representetiveness by location type and population groups

In [ ]:

df_home_users_geohashes_adm = df_home_users_geohashes.merge(
    df_geohash_city[["geohash", "SHEM_YISHUV_ENG", "YISHUV_STAT_2022", "SEMEL_YISHUV"]],
    on="geohash",
)

In [ ]:
df_city = df_geohash_city.drop_duplicates(["SEMEL_YISHUV"])

## In city scale

In [ ]:
df_homeloc_pop_cities = (
    df_home_users_geohashes_adm.groupby(["SEMEL_YISHUV", "month"])
    .size()
    .rename("home_locations")
    .reset_index()
    .merge(
        df_city[["pop_city", "flag_big_city", "flag_city", "SEMEL_YISHUV"]],
        on="SEMEL_YISHUV",
    )
)

In [ ]:
import matplotlib.pyplot as plt

# Extracting the first value of "home_locations" for each month and plotting
df_homeloc_pop_cities.groupby("month")[["pop_city", "home_locations"]].apply(
    lambda group: group[["pop_city", "home_locations"]].corr().iloc[0, 1]
).plot(figsize=(12, 6), marker="o", linewidth=2, color="teal", legend=False)

# Adding title and labels
plt.title(
    "Census Population vs Home locations in cities for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extracting the first value of "home_locations" for each month and plotting
correlations = (
    df_homeloc_pop_cities.groupby(["month", "flag_big_city"])[
        ["pop_city", "home_locations"]
    ]
    .apply(lambda group: group[["pop_city", "home_locations"]].corr().iloc[0, 1])
    .reset_index(name="correlation")
)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=correlations,
    x="month",
    y="correlation",
    hue="flag_big_city",
    marker="o",
    linewidth=2,
    color="teal",
    legend=True,
)

# Adding title and labels
plt.title(
    "Census Population vs Home locations in cities for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extracting the first value of "home_locations" for each month and plotting
correlations = (
    df_homeloc_pop_cities.groupby(["month", "flag_city"])[
        ["pop_city", "home_locations"]
    ]
    .apply(lambda group: group[["pop_city", "home_locations"]].corr().iloc[0, 1])
    .reset_index(name="correlation")
)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=correlations,
    x="month",
    y="correlation",
    hue="flag_city",
    marker="o",
    linewidth=2,
    color="teal",
    legend=True,
)

# Adding title and labels
plt.title(
    "Census Population vs Home locations in cities for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extracting the first value of "home_locations" for each month and plotting
correlations = (
    df_homeloc_pop_stat.groupby(["month", "flag_big_city"])[
        ["pop_approx", "home_locations"]
    ]
    .apply(lambda group: group[["pop_approx", "home_locations"]].corr().iloc[0, 1])
    .reset_index(name="correlation")
)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=correlations,
    x="month",
    y="correlation",
    hue="flag_big_city",
    marker="o",
    linewidth=2,
    color="teal",
    legend=True,
)

# Adding title and labels
plt.title(
    "Census Population vs Home locations in cities for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Extracting the first value of "home_locations" for each month and plotting
correlations = (
    df_homeloc_pop_stat.groupby(["month", "flag_city"])[
        ["pop_approx", "home_locations"]
    ]
    .apply(lambda group: group[["pop_approx", "home_locations"]].corr().iloc[0, 1])
    .reset_index(name="correlation")
)
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=correlations,
    x="month",
    y="correlation",
    hue="flag_city",
    marker="o",
    linewidth=2,
    color="teal",
    legend=True,
)

# Adding title and labels
plt.title(
    "Census Population vs Home locations in cities for Each Month",
    fontsize=16,
    fontweight="bold",
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Correlation", fontsize=12)

# Rotating x-ticks and improving visuals
plt.xticks(rotation=45, fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
df_homeloc_pop_stat["year"] = df_homeloc_pop_stat["month"].apply(lambda r: r[:4])

In [ ]:
annual_home_pop = (
    df_homeloc_pop_stat[(~df_homeloc_pop_stat.month.isin(["201911", "201909"]))]
    .groupby(["year", "YISHUV_STAT_2022", "flag_city"])[
        ["pop_approx", "home_locations"]
    ]
    .mean()
    .reset_index()
)

In [ ]:
from sklearn.linear_model import LinearRegression


def reg(df, x, y):
    y = df[y].values
    x = df[x].values.reshape(-1, 1)
    reg = LinearRegression(fit_intercept=True).fit(x, y)
    reg_line = reg.predict(x)
    return reg_line

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
import math

# Unique years
years = annual_home_pop.year.unique()

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

# Plot for each year
for ax, year in zip(axes, years[:3]):

    regline_city = reg(
        annual_home_pop[(annual_home_pop.year == year) & (annual_home_pop.flag_city)],
        x="home_locations",
        y="pop_approx",
    )
    regline_not_city = reg(
        annual_home_pop[(annual_home_pop.year == year) & (~annual_home_pop.flag_city)],
        x="home_locations",
        y="pop_approx",
    )
    y_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_city)
    ]["pop_approx"].values
    y_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_city)
    ]["pop_approx"].values
    x_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_city)
    ]["home_locations"].values
    x_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_city)
    ]["home_locations"].values
    r2_city = r2_score(y_city, regline_city)
    r2_not_city = r2_score(y_not_city, regline_not_city)
    sns.scatterplot(
        data=annual_home_pop[annual_home_pop.year == year],
        y="pop_approx",
        x="home_locations",
        hue="flag_city",
        ax=ax,
        alpha=0.5,
    )
    ax.set_title(str(year), fontsize=12)
    ax.set_ylabel("Population Approximation")
    ax.set_xlabel(
        "Home Locations" if ax == axes[0] else ""
    )  # Add ylabel only to the first subplot
    # Plot the regression line
    ax.plot(
        annual_home_pop[(annual_home_pop.year == year) & (annual_home_pop.flag_city)][
            "home_locations"
        ],
        regline_city,
        color="orange",
        alpha=0.5,
    )
    ax.text(
        math.floor(max(x_city) * 0.8),
        math.floor(max(regline_city)),
        f"R={round(r2_city,2)}",
        color="orange",
    )
    ax.plot(
        annual_home_pop[(annual_home_pop.year == year) & (~annual_home_pop.flag_city)][
            "home_locations"
        ],
        regline_not_city,
        color="blue",
        alpha=0.5,
    )
    ax.text(
        math.floor(max(x_not_city) * 0.8),
        math.floor(max(regline_not_city)),
        f"R={round(r2_not_city,2)}",
        color="Blue",
    )
# Adjust layout and show
plt.tight_layout()
plt.show()
# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

# Plot for each year
for ax, year in zip(axes, years[3:]):
    regline_city = reg(
        annual_home_pop[(annual_home_pop.year == year) & (annual_home_pop.flag_city)],
        x="home_locations",
        y="pop_approx",
    )
    regline_not_city = reg(
        annual_home_pop[(annual_home_pop.year == year) & (~annual_home_pop.flag_city)],
        x="home_locations",
        y="pop_approx",
    )
    y_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_city)
    ]["pop_approx"].values
    y_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_city)
    ]["pop_approx"].values
    x_city = annual_home_pop[
        (annual_home_pop.year == year) & (annual_home_pop.flag_city)
    ]["home_locations"].values
    x_not_city = annual_home_pop[
        (annual_home_pop.year == year) & (~annual_home_pop.flag_city)
    ]["home_locations"].values
    r2_city = r2_score(y_city, regline_city)
    r2_not_city = r2_score(y_not_city, regline_not_city)
    sns.scatterplot(
        data=annual_home_pop[annual_home_pop.year == year],
        y="pop_approx",
        x="home_locations",
        hue="flag_city",
        ax=ax,
        alpha=0.5,
    )
    ax.set_title(str(year), fontsize=12)
    ax.set_ylabel("Population Approximation")
    ax.set_xlabel(
        "Home Locations" if ax == axes[0] else ""
    )  # Add ylabel only to the first subplot
    ax.plot(
        annual_home_pop[(annual_home_pop.year == year) & (annual_home_pop.flag_city)][
            "home_locations"
        ],
        regline_city,
        color="orange",
        alpha=0.5,
    )
    ax.plot(
        annual_home_pop[(annual_home_pop.year == year) & (~annual_home_pop.flag_city)][
            "home_locations"
        ],
        regline_not_city,
        color="blue",
        alpha=0.5,
    )
    ax.text(
        math.floor(max(x_city) * 0.8),
        math.floor(max(regline_city)),
        f"R={round(r2_city,2)}",
        color="orange",
    )
    ax.text(
        math.floor(max(x_not_city) * 0.8),
        math.floor(max(regline_not_city)),
        f"R={round(r2_not_city,2)}",
        color="Blue",
    )
# Adjust layout and show
plt.tight_layout()
plt.show()

## Calculate model residuals

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd


# Function to calculate normalized RMSE for each neighborhood after building a model for the whole month
def calculate_rmse_by_neighborhood(group):
    # Fit regression model for the entire month
    y_month = group["pop_approx"].values # Target
    X_month = group["home_locations"].values.reshape(-1, 1) # Predictor  

    model = LinearRegression()
    model.fit(X_month, y_month)
    y_pred_month = model.predict(X_month)

    # Add predictions to the group for calculating RMSE by neighborhood
    group["y_pred"] = y_pred_month

    group["relative_error"] = abs(group["pop_approx"] - group["y_pred"])


    # Return as a DataFrame for clarity
    return group[["YISHUV_STAT_2022","relative_error","y_pred"]]


# Group by month and calculate RMSE for each neighborhood
results = df_homeloc_pop_stat.groupby("month").apply(calculate_rmse_by_neighborhood)

# Resetting index for cleaner DataFrame
results_df = results.reset_index()

# Group by month and calculate RMSE for each neighborhood
results_hiloni = (
    df_homeloc_pop_stat[df_homeloc_pop_stat.flag_not_religious]
    .groupby("month")
    .apply(calculate_rmse_by_neighborhood)
)

# Resetting index for cleaner DataFrame
results_df_hiloni = results_hiloni.reset_index()

# Display the results

## Merge all together

In [ ]:
df_homeloc_pop_stat = df_homeloc_pop_stat.drop(['relative_error'], axis=1).merge(
    results_df.drop(["level_1"], axis=1), on=["YISHUV_STAT_2022", "month"], how="left"
)

In [ ]:
df_homeloc_pop_stat = df_homeloc_pop_stat.drop(['relative_error_not_relig_only'], axis=1).merge(
    results_df_hiloni.drop(["level_1"], axis=1),
    on=["YISHUV_STAT_2022", "month"],
    how="left",
    suffixes=("","_not_relig_only")
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Group the data and reset the index for Seaborn
grouped_data = (
    df_homeloc_pop_stat[df_homeloc_pop_stat["flag_big_city"]]
    .groupby(["SHEM_YISHUV_ENG", "religion_level"])
    .size()
    .reset_index(name="count")
)

# Set the Seaborn style
sns.set(style="whitegrid")

# Create the bar plot
plt.figure(figsize=(14, 6))
sns.barplot(
    data=grouped_data,
    x="SHEM_YISHUV_ENG",
    y="count",
    hue="religion_level",
    palette="Set2",
)

# Add title and labels
plt.title(
    "Distribution by City and Religious Affiliation", fontsize=16, fontweight="bold"
)
plt.xlabel("City", fontsize=12)
plt.ylabel("Count", fontsize=12)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=10, ha="right")

# Adjust the legend
plt.legend(
    title="Religious Affiliation", fontsize=10, title_fontsize=12, loc="upper right"
)

# Adjust layout and display
plt.tight_layout()
plt.show()

Apparently arabs are classified as traditional ( masorati)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate mean relative error for big cities
mean_city_error = (
    df_homeloc_pop_stat[df_homeloc_pop_stat["flag_big_city"]]
    .groupby(["SHEM_YISHUV_ENG", "month"])["relative_error"]
    .mean()
    .reset_index()
)

# Set the Seaborn style
sns.set(style="whitegrid")

# Plot with grey colors and different line styles
plt.figure(figsize=(14, 6))
sns.lineplot(
    data=mean_city_error,
    x="month",
    y="relative_error",
    hue="SHEM_YISHUV_ENG",
    palette="gray",  # Use grey color palette
    style="SHEM_YISHUV_ENG",  # Different line styles for cities
    dashes=True,
)

# Customize the title and labels
plt.title("Mean  Error by City Over Months", fontsize=16, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Mean  Error", fontsize=12)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)

# Adjust legend
plt.legend(title="City", fontsize=10, title_fontsize=12, loc="upper left")

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate mean relative error for big cities
mean_city_error = (
    df_homeloc_pop_stat[
        (df_homeloc_pop_stat["flag_big_city"])
        & (df_homeloc_pop_stat.flag_not_religious)
    ]
    .groupby(["SHEM_YISHUV_ENG", "month"])["relative_error_not_relig_only"]
    .mean()
    .reset_index()
)

# Set the Seaborn style
sns.set(style="whitegrid")

# Plot with grey colors and different line styles
plt.figure(figsize=(14, 6))
sns.lineplot(
    data=mean_city_error,
    x="month",
    y="relative_error_not_relig_only",
    hue="SHEM_YISHUV_ENG",
    palette="Set2",  # Use grey color palette
    style="SHEM_YISHUV_ENG",  # Different line styles for cities
    dashes=True,
)

# Customize the title and labels
plt.title("Mean Error by City Over Months", fontsize=16, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Mean  Error", fontsize=12)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)

# Adjust legend
plt.legend(title="City", fontsize=10, title_fontsize=12, loc="upper right")

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
city_error_avg = df_homeloc_pop_stat[(df_homeloc_pop_stat["flag_big_city"])&(df_homeloc_pop_stat.flag_not_religious)].groupby(
    ["SHEM_YISHUV_ENG", "month"]
)['relative_error_not_relig_only'].mean().groupby("SHEM_YISHUV_ENG").mean()/df_homeloc_pop_stat[(df_homeloc_pop_stat["flag_big_city"])&(df_homeloc_pop_stat.flag_not_religious)].groupby(
    ["SHEM_YISHUV_ENG"])['pop_approx'].mean()
city_error_avg.sort_values()

In [ ]:
list_city_fine = city_error_avg[city_error_avg<1.5].index


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate mean relative error for big cities
mean_city_error = (
    df_homeloc_pop_stat[
        (df_homeloc_pop_stat["flag_big_city"])
        & (df_homeloc_pop_stat.flag_not_religious)
        & (df_homeloc_pop_stat["SHEM_YISHUV_ENG"].isin(list_city_fine ))
    ]
    .groupby(["SHEM_YISHUV_ENG", "month"])["relative_error_not_relig_only"]
    .mean()
    .reset_index()
)

# Set the Seaborn style
sns.set(style="whitegrid")

# Plot with grey colors and different line styles
plt.figure(figsize=(14, 6))
sns.lineplot(
    data=mean_city_error,
    x="month",
    y="relative_error_not_relig_only",
    hue="SHEM_YISHUV_ENG",
    palette="Set2",  # Use grey color palette
    style="SHEM_YISHUV_ENG",  # Different line styles for cities
    dashes=True,
)

# Customize the title and labels
plt.title("Mean Relative Error by City Over Months", fontsize=16, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Mean Relative Error", fontsize=12)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)

# Adjust legend
plt.legend(title="City", fontsize=10, title_fontsize=12, loc="upper left")

# Adjust layout and display
plt.tight_layout()
plt.show()